# Husky 와이어태핑을 통한 부채널 파형 수집

## 시나리오
- 통신 데이터 교환 : **타겟 보드(CW308_STM32F3)** 및 **ChipWhisperer Lite**는 호스트 PC 와 평문/암호문을 주고받음.
- **ChipWhisperer Husky** 는 부채널 파형 수집을 위해 타겟 보드에 **와이어태핑** 수행
  > 단, 실험의 편의성을 위해 타겟 보드에서 암호화 구간에 트리거 신호 생성 (실제 공격에서는 통신신호 등을 트리거 신호로 이용해야 함)
  - **트리거** : 타겟 보드의 트리거 라인을 Husky 전면 20-pin의 **D0** 에 연결 (CW308 보드의 GPIO4/TRIG)
  - **클럭**   : 타겟 보드의 클럭을 Husky 전면 **AUX MCX** 에 연결 (정확한 주파수 미상 → 카운터로 탐색) (CW308 보드의 CLKIN)
  - **전압**   : 타겟 보드의 션트 양단을 Husky 측면 **Measure (Pos)** 에 연결 (CW308 보드의 SHUNTL)

## 노트북 구성
1. 라이브러리 import & 멀티 디바이스(Lite + Husky) 연결
2. Lite 를 통한 타겟 보드 통신 채널 확보 (SimpleSerial2)
3. 타겟 펌웨어 빌드 및 프로그래밍
4. 통신 정상 동작 검증 (Golden 모델 비교)
5. **Husky 측 부채널 파형 수집 환경 구성** (트리거 / 클럭 / ADC)
6. `Encrypt()` 추상화 함수 정의
7. 파형 수집 루프 실행
8. 수집 결과 시각화 및 자원 해제

## 1. 라이브러리 import & 상수 정의

`%run My_script.ipynb` 은 워크스페이스에 정의된 헬퍼(`my_fsr_cmd` 등)와 그래프 패키지를 사전 로드함.

In [1]:
# 사전 정의된 헬퍼 (my_fsr_cmd, 그래프 유틸 등)를 로드
%run My_script.ipynb

import chipwhisperer as cw

# ─────────────────────────────────────────
# 타겟 / 펌웨어 관련 상수
# ─────────────────────────────────────────
PLATFORM      = 'CW308_STM32F3'   # 타겟 보드 종류
SCOPETYPE     = 'OPENADC'         # 캡처 장치 (Lite/Husky 공통)
CRYPTO_TARGET = 'NONE'            # 사용 암호 라이브러리 (없음 -> 자체 펌웨어)
SS_VER        = 'SS_VER_2_1'      # SimpleSerial 프로토콜 버전

Loading BokehJS ...

## 2. ChipWhisperer 다중 장치 연결 관리자

호스트 PC 에 동시에 연결된 모든 ChipWhisperer 장치(Lite, Husky)를 일괄 검출/연결/해제.

In [2]:
def connect_all_devices() -> dict:
    """연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환"""
    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}

    for device in device_list:
        # 'ChipWhisperer-Lite' → 'ChipWhisperer_Lite' 처럼 dict 키로 쓰기 쉽게 변환
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes

# PC에 연결된 모든 장치 인식 및 할당
scopes = connect_all_devices()
lite_scope = scopes["ChipWhisperer_Lite"]
husky_scope = scopes["ChipWhisperer_Husky"]

발견된 장치 수: 2

  [✓] ChipWhisperer_Husky 연결 완료  (SN: 502032204c5846303130313137313032)
  [✓] ChipWhisperer_Lite 연결 완료  (SN: 44203120394d36433130322030313035)


## 3. Lite 를 통한 타겟 보드 통신 채널 확보

타겟 보드는 Lite 와 UART(SimpleSerial2)로 연결되어 있음. 이를 통해 호스트 PC ↔ 타겟 평문/암호문 통신을 수행.

In [3]:
# Lite를 통한 타겟 보드 연결 설정
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("지원되지 않는 SimpleSerial 버전입니다.")

try:
    target = cw.target(lite_scope, target_type)
    print("\n[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공")
except Exception as e:
    print(f"\n[✗] ChipWhisperer_Lite에 타겟 보드 연결 실패: {e}")


[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공


## 4. 타겟 펌웨어 빌드 (Makefile) & 타겟 보드에 펌웨어 프로그래밍 (Lite 경유) & 빌드 산출물 정리

- `simpleserial_main/` 디렉터리의 펌웨어 소스를 지정 플랫폼으로 컴파일.
- `lite_scope.default_setup()` : Lite 의 클럭/UART 핀 등을 표준 설정으로 초기화  
- 이 단계에서 변경되는 클럭/IO 설정은 **Lite 측 설정**이며, Husky 의 ADC/트리거 설정과는 **독립적**임.

In [4]:
# 1. 펌웨어 컴파일 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 완료")

# 2. 프로그래머 설정
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("프로그래머가 지원되지 않는 플랫폼입니다.")

# 3. 펌웨어 플래싱 (Lite가 프로그래머 역할을 수행)
lite_scope.default_setup()
try:
    hex_path = f"simpleserial_main/simpleserial-base-{PLATFORM}.hex"
    cw.program_target(lite_scope, prog, hex_path)
    print(f"[✓] {PLATFORM} 타겟 보드에 프로그램 업로드 완료")
except Exception as e:
    print(f"[✗] 펌웨어 프로그램 실패: {e}")

# 4. 펌웨어 컴파일 클린 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 클린 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}", "clean"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 클린 완료")

펌웨어 컴파일 중...
펌웨어 컴파일 완료
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 71859277                  to 93308112                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538459                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538459.0           

## 5. 통신 동작 검증 (Golden 모델 비교)

본격 파형 수집 전, 타겟이 정상적으로 평문/키를 받고 결과(k ⊕ p)를 반환하는지 점검.

In [5]:
MAX_DATA_LEN = 50

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(k), 평문(p) 생성 후 호스트에서 사전 계산한 골든 결과(k XOR p)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

# 0x81 = 데이터 전송 명령 ('k'=key, 'p'=plaintext, 'l'=length)
# 0x82 = 연산 트리거 명령
# 0x83 = 결과 회수 명령
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과 : {Return_k_XOR_p.hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('✅ 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    print('❌ 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

=== 결과 비교 ===
타겟 결과 : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc
골든 모델 : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc

✅ 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)


## 6. Husky 측 부채널 파형 수집 환경 구성

Lite/타겟 사이의 동작은 완전히 정상이라고 가정한 상태에서, **Husky 는 단지 와이어태핑 관측자**로서 동작하도록 설정함.  
이하의 모든 설정은 `husky_scope` 에만 적용되며 Lite 설정과는 독립적임.

In [6]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
# PLL 입력 소스를 외부 클럭(extclk)으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
# AUX MCX 를 입력(high-Z)으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'

print(f"clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
print(f"io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_shift_steps           changed from 0                         to 4592                     
scope.trace.capture

(ChipWhisperer Scope ERROR|File ChipWhispererHuskyClock.py:1242) Failed to update clkgen_freq: Could not calculate pll settings for input 7363636.363636363, output 7363636.363636363 with mul 4


clock.clkgen_src  = system
io.aux_io_mcx     = high_z


In [7]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
# 주파수 카운터의 측정 대상을 외부 클럭으로 지정
husky_scope.clock.freq_ctr_src = 'extclk'
# 카운터 초기 안정화를 위한 짧은 대기
time.sleep(0.5)

# 내부/외부 클럭 주파수 동기화
husky_scope.clock.clkgen_freq = husky_scope.clock.freq_ctr
# ADC 샘플레이트 = 타겟 클럭 × 4 (4× 오버샘플링)
husky_scope.clock.adc_mul = 4
# ADC 리셋
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("  - ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    print("  ! 경고: ADC 클럭 동기화 실패 (Lock Error)")

if husky_scope.clock.clkgen_locked:
    print("✅ Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    print("⚠ Husky PLL 잠금 실패! 외부 클럭의 진폭/듀티/안정성을 확인하세요.")

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:663) 
                Could not calculate pll settings for the requested frequency (7384506); 
                generating a 7400000 clock instead.
                It may be possible to get closer to the requested frequency
                with a different adc_mul.
                It may also be possible to get closer to the requested
                frequency if you set scope.clock.pll._allow_rdiv to True;
                however this can result in an inconsistant clock phase between
                the target and ADC clocks; use at your own risk!
                
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:663) 
                Could not calculate pll settings for the requested frequency (7384506); 
                generating a 7400000 clock instead.
                It may be possible to get closer to the requested frequency
                with a different adc_mul.
                It may also be possible to

  - ADC 클럭 동기화 완료
   - ADC 샘플레이트 (adc_freq): 29,600,000 Hz
✅ Husky PLL 잠금 성공
   - 타겟 클럭 (clkgen_freq) : 7,400,000 Hz


In [8]:
# ---------------------------------------------------------
# [3] 트리거 핀 설정 (전면 USERIO - D0 핀)
# ---------------------------------------------------------
# 트리거 입력 소스 = 전면 USERIO D0
husky_scope.trigger.triggers = 'userio_d0'
# 단순 엣지/레벨 검출용 'basic' 트리거 모듈 사용
husky_scope.trigger.module = 'basic'
# 캡처 시작 조건 = 상승 엣지 (타겟이 트리거를 LOW → HIGH 로 토글)
husky_scope.adc.basic_mode = 'rising_edge'

print("[✓] Husky 스코프 파라미터 설정 완료")
print(f"trigger.triggers = {husky_scope.trigger.triggers}")
print(f"trigger.module   = {husky_scope.trigger.module}")
print(f"adc.basic_mode   = {husky_scope.adc.basic_mode}")

[✓] Husky 스코프 파라미터 설정 완료
trigger.triggers = userio_d0
trigger.module   = basic
adc.basic_mode   = rising_edge


### ADC 캡처 파라미터 설정 (게인 / 샘플 수 / 오프셋)

측면 Measure (Pos/Neg) 차동 입력이 곧 Husky ADC 의 아날로그 경로이므로,  
별도 라우팅 없이 **게인** 과 **샘플 수** 만 적절히 설정하면 됨.

In [9]:
# LNA 게인 (dB). 보통 20~30 dB 사이. 너무 높으면 클리핑, 너무 낮으면 SNR 저하.
husky_scope.gain.db = 25

# 한 번의 캡처에서 수집할 샘플 개수
husky_scope.adc.samples = 10000

# 트리거 이후 캡처 시작점 (0 = 트리거 즉시 캡처 시작)
husky_scope.adc.offset = 0

# 트리거 이전 샘플 (사전 캡처). 필요 시 양수로 설정 가능.
husky_scope.adc.presamples = 0

print(f"gain.db          = {husky_scope.gain.db}")
print(f"adc.samples      = {husky_scope.adc.samples}")
print(f"adc.offset       = {husky_scope.adc.offset}")
print(f"adc.presamples   = {husky_scope.adc.presamples}")

gain.db          = 25.091743119266056
adc.samples      = 10000
adc.offset       = 0
adc.presamples   = 0


In [10]:
# 비교용 (cw-lite)
lite_scope.clock.adc_mul = 4
lite_scope.clock.reset_adc()
lite_scope.gain.db = 25
lite_scope.adc.samples = 10000
lite_scope.adc.offset = 0
lite_scope.adc.presamples = 0

print(f"gain.db          = {lite_scope.gain.db}")
print(f"adc.samples      = {lite_scope.adc.samples}")
print(f"adc.offset       = {lite_scope.adc.offset}")
print(f"adc.presamples   = {lite_scope.adc.presamples}")

gain.db          = 24.8359375
adc.samples      = 10000
adc.offset       = 0
adc.presamples   = 0


## 7. 다수 파형 수집 루프

In [11]:
def Encrypt(data_k, data_p):    
    my_fsr_cmd(target, 0x81, 'k', data_k)
    my_fsr_cmd(target, 0x81, 'p', data_p)
    my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
    my_fsr_cmd(target, 0x82, 'c', [])  
    return my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

# 수집 데이터 초기화
t_husky = []
t_lite = []
i_k = []
i_p = []
o_c = []

N_TRACES = 10 # 수집할 총 파형 개수
print(f"\n=== [{N_TRACES}]개의 파형 수집을 시작합니다 ===")

# 재현성을 위해 시드 고정
MAX_DATA_LEN = 50  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)
random.seed(1)

for i in range(N_TRACES):
    # 1. 랜덤 생성
    data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
    data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
    
    # 2. Husky 스코프를 Arm 상태로 전환하여 D0 핀의 트리거 신호 대기
    husky_scope.arm()
    lite_scope.arm()
    
    # 3. Lite 스코프를 통해 타겟 보드로 통신 (이 과정에서 타겟이 트리거 발생)
    ct = Encrypt(data_k, data_p)
    
    # 4. Husky가 트리거를 성공적으로 인식했는지 타임아웃 여부 점검
    ret = husky_scope.capture()
    lite_scope.capture()
    if ret:
        print(f"  [✗] 파형 {i} 수집 실패: 타임아웃 발생! 타겟 펌웨어 혹은 D0 핀 연결 상태를 확인하세요.")
        continue
        
    # 5. 수집된 파형 데이터 추출 및 리스트 저장
    wave_husky = husky_scope.get_last_trace()
    wave_lite = lite_scope.get_last_trace()
    
    t_husky.append(wave_husky)
    t_lite.append(wave_lite)
    i_k.append(data_k)
    i_p.append(data_p)
    o_c.append(ct)
    
    # 10개 단위로 진행 상황 모니터링
    if (i + 1) % 10 == 0:
        print(f"  - 진행 상황: {i + 1} / {N_TRACES} 캡처 완료")

print("\n[✓] 모든 파형 수집이 성공적으로 완료되었습니다!")


=== [10]개의 파형 수집을 시작합니다 ===
  - 진행 상황: 10 / 10 캡처 완료

[✓] 모든 파형 수집이 성공적으로 완료되었습니다!


In [12]:
# ── 다중 파형 Bokeh 시각화 ──────────────────────────────
NUM_PLOT = 10  # 겹쳐 그릴 파형 수

t = np.array(t_husky)
x = np.arange(t.shape[1])

p = figure(
    width=900, height=360,
    title=f'Power Traces Overlay (first {NUM_PLOT} traces)',
    x_axis_label='Sample Index',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

palette = Category10[10]
for i in range(NUM_PLOT):
    p.line(x, t[i],
           line_width=1.0,
           line_color=palette[i % 10],
           line_alpha=0.75,
           legend_label=f'Trace {i}')

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
p.outline_line_color = None

# 범례 설정
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'  # 범례 클릭 시 해당 파형 숨김
p.legend.label_text_font_size = '9pt'
p.legend.background_fill_alpha = 0.7

# Hover 툴팁
hover = HoverTool(tooltips=[('Sample', '$x{0}'), ('Amplitude', '$y{0.0000}')])
p.add_tools(hover)

show(p)
print('\n완료!')


완료!


In [13]:
# ── 다중 파형 Bokeh 시각화 ──────────────────────────────
NUM_PLOT = 10  # 겹쳐 그릴 파형 수

t = np.array(t_lite)
x = np.arange(t.shape[1])

p = figure(
    width=900, height=360,
    title=f'Power Traces Overlay (first {NUM_PLOT} traces)',
    x_axis_label='Sample Index',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

palette = Category10[10]
for i in range(NUM_PLOT):
    p.line(x, t[i],
           line_width=1.0,
           line_color=palette[i % 10],
           line_alpha=0.75,
           legend_label=f'Trace {i}')

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
p.outline_line_color = None

# 범례 설정
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'  # 범례 클릭 시 해당 파형 숨김
p.legend.label_text_font_size = '9pt'
p.legend.background_fill_alpha = 0.7

# Hover 툴팁
hover = HoverTool(tooltips=[('Sample', '$x{0}'), ('Amplitude', '$y{0.0000}')])
p.add_tools(hover)

show(p)
print('\n완료!')


완료!


In [14]:
def disconnect_all_devices(scopes: dict) -> None:
    # 타겟 객체 먼저 닫기 (Lite 의 UART 점유 해제)
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")

    """딕셔너리 내 모든 장치 연결 해제"""
    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
    scopes.clear()

# 파형 수집 및 데이터 저장이 끝난 후 반드시 포트 및 메모리 자원 반환
disconnect_all_devices(scopes)

  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료

장치 연결 해제 중...
  [✓] ChipWhisperer_Husky 연결 해제 완료
  [✓] ChipWhisperer_Lite 연결 해제 완료
